# TP Integrador — Semana 2: Preparación del Dataset con PyTorch

**Materia:** Redes Neuronales Profundas — Ingeniería en Sistemas de Información  
**Dataset:** Team Separation (Animals) — Roboflow Universe v5  
**URL:** https://universe.roboflow.com/animals-67mq4/team-separation/dataset/5

Este notebook cubre todo el pipeline de preparación de datos:
- Descarga y exploración del dataset
- Generación de CSVs por split
- Implementación de `AnimalDataset` (PyTorch `Dataset`)
- Definición de transforms y augmentations
- Configuración de `DataLoader`s
- Verificación visual del batch de entrenamiento

## Sección 0 — Setup e Instalación de Dependencias

Se importan todas las bibliotecas necesarias y se fija la semilla global (`SEED = 42`) para garantizar reproducibilidad en todos los pasos que involucren aleatoriedad (shuffling, augmentations, selección de ejemplos).

In [ ]:
# Descomentar si se ejecuta en un entorno nuevo o Google Colab:
# !pip install roboflow torch torchvision matplotlib seaborn pandas numpy Pillow

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ── Semilla global para reproducibilidad ──────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"Semilla:      {SEED}")

In [ ]:
# ── Detección de entorno (local vs Google Colab) ──────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # En Colab: clonar el repositorio o montar Drive y ajustar BASE_DIR
    BASE_DIR = Path("/content/tp-integrador")
else:
    # Local: el notebook está en dev/, los datos en data/
    BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
DATA_DIR = BASE_DIR / "data"
print(f"BASE_DIR: {BASE_DIR}")
print(f"RAW_DIR:  {RAW_DIR}")

## Sección 1 — Descarga del Dataset

Se descarga el dataset desde Roboflow Universe usando la API key (leída desde la variable de entorno `ROBOFLOW_API_KEY`). Si el dataset ya existe en `data/raw/`, se omite la descarga para no repetir el proceso.

El formato `"folder"` descarga las imágenes organizadas en subcarpetas por clase, lo que es ideal para tareas de clasificación.

In [ ]:
if not RAW_DIR.exists() or not any(RAW_DIR.rglob("*.jpg")):
    print("Descargando dataset...")
    import subprocess
    subprocess.run([sys.executable, str(DATA_DIR / "download_dataset.py")], check=True)
else:
    print("Dataset ya disponible.")

# Mostrar estructura de carpetas resultante
print()
for split in ["train", "valid", "test"]:
    split_path = RAW_DIR / split
    if split_path.exists():
        clases = [d.name for d in sorted(split_path.iterdir()) if d.is_dir()]
        print(f"\nSplit '{split}' -> {len(clases)} clases: {clases}")
        for cls in clases:
            n = len(list((split_path / cls).glob("*.jpg")))
            print(f"   {cls}/: {n} imagenes")

## Sección 2 — Exploración del Dataset

Antes de entrenar cualquier modelo, es fundamental entender la distribución de los datos. Aquí analizamos:
- Cuántas clases hay y cómo se llaman (detectadas dinámicamente)
- La distribución de imágenes por clase y por split
- Ejemplos visuales reales del dataset
- Información técnica de las imágenes (resolución, canales, formato)

### 2.1 — Clases y distribución por split

In [ ]:
# Construir DataFrame de distribución detectando clases dinámicamente
registros = []
SPLITS = {"train": "train", "valid": "valid", "test": "test"}

for split_name, split_folder in SPLITS.items():
    split_path = RAW_DIR / split_folder
    if not split_path.exists():
        continue
    for clase_dir in sorted(split_path.iterdir()):
        if clase_dir.is_dir():
            n = len(list(clase_dir.glob("*")))
            registros.append({"split": split_name, "clase": clase_dir.name, "n_imagenes": n})

df_dist = pd.DataFrame(registros)

print("Tabla de distribucion por clase y split:")
print(df_dist.pivot(index="clase", columns="split", values="n_imagenes").fillna(0).astype(int).to_string())
print(f"\nTotal de clases: {df_dist['clase'].nunique()}")
print(f"Total de imagenes: {df_dist['n_imagenes'].sum()}")

In [ ]:
# Gráfico de barras agrupadas por clase y split
pivot = df_dist.pivot(index="clase", columns="split", values="n_imagenes").fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", ax=ax, rot=45)
ax.set_title("Distribucion de imagenes por clase y split")
ax.set_ylabel("Numero de imagenes")
ax.set_xlabel("Clase")
ax.legend(title="Split")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=100)
plt.show()
print("Guardado: dev/class_distribution.png")

**Analisis de distribucion:**

Observar el grafico anterior para responder:
- ¿Hay desbalance de clases? Si algunas clases tienen muchas mas imagenes que otras, el modelo tendra tendencia a predecirlas mas frecuentemente.
- ¿Es severo el desbalance? Si una clase tiene menos del 20% de la clase mas grande, puede requerir tecnicas como class weighting o oversampling.
- En la Semana 3 (entrenamiento) se podra usar `WeightedRandomSampler` de PyTorch para mitigar el desbalance si es necesario.

### 2.2 — Ejemplos visuales del dataset

Mostrar al menos 2 imagenes por clase para tener una intuicion visual del tipo de datos con el que trabajamos. Esto permite identificar:
- Variabilidad de poses, iluminacion y fondo
- Calidad general de las imagenes
- Posibles ambiguedades entre clases

In [ ]:
# Mostrar grilla de ejemplos: al menos 2 por clase
train_path = RAW_DIR / "train"
clases = sorted([d.name for d in train_path.iterdir() if d.is_dir()])

N_POR_CLASE = 3
n_clases = len(clases)
fig, axes = plt.subplots(n_clases, N_POR_CLASE, figsize=(4 * N_POR_CLASE, 4 * n_clases))

if n_clases == 1:
    axes = axes[np.newaxis, :]

for row, cls in enumerate(clases):
    cls_path = train_path / cls
    imagenes = sorted(cls_path.glob("*.jpg"))[:N_POR_CLASE]
    for col, img_path in enumerate(imagenes):
        img = Image.open(img_path).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{cls}\n{img_path.name}", fontsize=7)
        axes[row, col].axis("off")
    for col in range(len(imagenes), N_POR_CLASE):
        axes[row, col].axis("off")

plt.suptitle("Ejemplos visuales del dataset (split train, 3 por clase)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Clases mostradas: {clases}")

**Observaciones visuales:**
- Completar tras ejecutar: resolución típica observada, variabilidad de poses y fondos, condiciones de captura (exterior/interior, iluminación), posibles ambigüedades entre clases.

### 2.3 — Información técnica de las imágenes

Analizar la distribucion de resoluciones y relaciones de aspecto en una muestra aleatoria. Es importante saberlo antes de definir los transforms, ya que todas las imagenes se redimensionaran a 224x224.

In [ ]:
# Muestra aleatoria de 50 imagenes del split train
todas_las_imagenes = list((RAW_DIR / "train").rglob("*.jpg"))
random.seed(SEED)
muestra = random.sample(todas_las_imagenes, min(50, len(todas_las_imagenes)))

infos = []
for img_path in muestra:
    with Image.open(img_path) as img:
        w, h = img.size
        infos.append({
            "width": w, "height": h,
            "aspect_ratio": round(w / h, 3),
            "channels": len(img.getbands()),
            "format": img.format or img_path.suffix.upper(),
        })

df_info = pd.DataFrame(infos)
print("Estadisticas de la muestra (50 imagenes):")
print(df_info[["width", "height", "aspect_ratio"]].describe().round(2).to_string())
print(f"\nCanales unicos: {df_info['channels'].unique()}")
print(f"Formatos unicos: {df_info['format'].unique()}")

# Resoluciones unicas
resoluciones = df_info.apply(lambda r: f"{int(r.width)}x{int(r.height)}", axis=1).value_counts()
print(f"\nResoluciones encontradas:")
print(resoluciones.to_string())

In [ ]:
# Histograma de relacion de aspecto
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_info["aspect_ratio"], bins=20, edgecolor="black", color="steelblue")
ax.set_title("Distribucion de relacion de aspecto (W/H)")
ax.set_xlabel("Aspecto (W/H)")
ax.set_ylabel("Frecuencia")
ax.axvline(1.0, color="red", linestyle="--", label="Cuadrado (1:1)")
ax.legend()
plt.tight_layout()
plt.show()

## Sección 3 — Particionado y Generación de CSVs

### Justificacion de los splits originales de Roboflow

Se respetan los splits definidos por los creadores del dataset por las siguientes razones:

1. **Estratificacion garantizada:** Los splits de Roboflow aseguran que todas las clases esten representadas en train, valid y test en proporciones similares.
2. **Reproducibilidad:** Re-particionar romperia la comparabilidad con otros trabajos y papers que usen el mismo dataset.
3. **Evitar data leakage:** Al no re-mezclar, no existe riesgo de que imagenes del mismo animal aparezcan en train y test.
4. **Semilla SEED=42:** Garantiza reproducibilidad en los pasos que si tienen aleatoriedad (augmentations, batch shuffling).

In [ ]:
SPLIT_MAP = {"train": "train", "val": "valid", "test": "test"}

# Detectar clases dinamicamente desde el split train
clases = sorted([d.name for d in (RAW_DIR / "train").iterdir() if d.is_dir()])
clase_to_idx = {cls: idx for idx, cls in enumerate(clases)}
print("Clases detectadas:", clases)
print("Mapeo a indices: ", clase_to_idx)

for csv_name, split_folder in SPLIT_MAP.items():
    registros = []
    split_path = RAW_DIR / split_folder
    for clase_dir in sorted(split_path.iterdir()):
        if not clase_dir.is_dir():
            continue
        for img_path in sorted(clase_dir.glob("*")):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                registros.append({
                    "filepath": str(img_path.relative_to(BASE_DIR)),
                    "label": clase_dir.name,
                    "label_idx": clase_to_idx[clase_dir.name],
                    "split": csv_name,
                })
    df = pd.DataFrame(registros)
    output_path = DATA_DIR / f"{csv_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"\n{csv_name}.csv guardado — {len(df)} filas")
    print(df.groupby("label")["label_idx"].count().to_string())

In [ ]:
# Tabla resumen de la particion
for csv_name in SPLIT_MAP:
    df = pd.read_csv(DATA_DIR / f"{csv_name}.csv")
    print(f"\n{csv_name}.csv — primeras 3 filas:")
    print(df.head(3).to_string())
    print(f"Total: {len(df)} imagenes")

## Sección 4 — Clase Dataset de PyTorch

Se implementa `AnimalDataset`, una subclase de `torch.utils.data.Dataset`. Esto permite integrar el dataset en el ecosistema de PyTorch y usar `DataLoader` para batching, shuffling y carga paralela.

### Por que usar AnimalDataset en lugar de ImageFolder?

| Criterio | `AnimalDataset` (CSV) | `ImageFolder` |
|---|---|---|
| Flexibilidad de estructura | Alta — solo necesita un CSV | Baja — requiere `split/clase/imagen` exacto |
| Portabilidad | Alta — el CSV es autocontenido | Media — depende de la estructura de carpetas |
| Facilidad de cambio | Cambiar el CSV es suficiente | Hay que reorganizar carpetas |
| Semanas siguientes | Facil de extender con metadatos | Mas rigido |

Se elige `AnimalDataset` para mayor control y portabilidad en las etapas futuras del proyecto.

In [ ]:
class AnimalDataset(Dataset):
    """Dataset personalizado para el dataset de separacion de animales.

    Lee imagenes desde un CSV con columnas 'filepath' y 'label_idx'.
    Aplica las transforms indicadas al instanciar.

    Args:
        csv_path  (str | Path): Ruta al CSV del split.
        transform (callable):   Transforms de torchvision.
        base_dir  (str | Path): Directorio base para rutas relativas.
    """

    def __init__(self, csv_path, transform=None, base_dir=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform
        self.base_dir = Path(base_dir) if base_dir else BASE_DIR
        self.classes = sorted(self.data["label"].unique().tolist())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self.base_dir / row["filepath"]
        image = Image.open(img_path).convert("RGB")
        label = int(row["label_idx"])
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_names(self):
        return self.classes

In [ ]:
# Instanciar SIN transforms para verificar que lee correctamente
raw_train = AnimalDataset(DATA_DIR / "train.csv")
raw_val   = AnimalDataset(DATA_DIR / "val.csv")
raw_test  = AnimalDataset(DATA_DIR / "test.csv")

print(f"Train: {len(raw_train)} imagenes")
print(f"Val:   {len(raw_val)} imagenes")
print(f"Test:  {len(raw_test)} imagenes")
print(f"Clases: {raw_train.get_class_names()}")

# Acceder a un elemento aleatorio
idx = random.randint(0, len(raw_train) - 1)
img, label = raw_train[idx]
print(f"\nEjemplo aleatorio [idx={idx}]:")
print(f"  Tipo: {type(img)}, Tamanio: {img.size}, Clase: {raw_train.classes[label]}")

plt.imshow(img)
plt.title(f"Clase: {raw_train.classes[label]} (idx={label})")
plt.axis("off")
plt.show()

## Sección 5 — Preprocesamiento y Transforms

### 5.1 — Constantes

Se usa `IMG_SIZE = 224` porque ResNet50 (el modelo preentrenado objetivo) fue diseñado para recibir imágenes de 224×224 píxeles.

Las estadísticas de normalización (`IMAGENET_MEAN` y `IMAGENET_STD`) provienen del dataset ImageNet con el que fue entrenado ResNet50. Sus filtros aprendidos esperan datos en ese rango: usar otras estadísticas desplazaría la distribución y degradaría la transferencia de conocimiento (transfer learning).

In [ ]:
IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Tamanio de entrada al modelo: {IMG_SIZE}x{IMG_SIZE}")
print(f"Media ImageNet:               {IMAGENET_MEAN}")
print(f"Desv. est. ImageNet:          {IMAGENET_STD}")

### 5.2 — Transforms de entrenamiento (con Data Augmentation)

La augmentation solo se aplica al split de train. Su objetivo es artificialmente aumentar la variabilidad de los datos de entrenamiento para mejorar la generalización del modelo.

| Augmentation | Justificacion para imagenes de animales |
|---|---|
| `RandomResizedCrop` | Simula diferentes distancias de la camara al animal y distintos encuadres |
| `RandomHorizontalFlip` | Los animales pueden aparecer mirando en cualquier direccion horizontal |
| `RandomRotation(15)` | Pequenias variaciones de inclinacion de la camara son realistas |
| `ColorJitter` | Variaciones de iluminacion natural: amanecer, noche, sombras, distintas epocas del anio |
| ~~`RandomVerticalFlip`~~ | **NO aplicar**: los animales siempre tienen las patas abajo |
| ~~`Rotacion > 30`~~ | **NO aplicar**: distorsionaria la postura natural del animal |

In [ ]:
# Transforms de entrenamiento: Resize -> Crop aleatorio -> Flips -> Rotacion -> Color -> Tensor -> Normalize
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Pipeline de train (CON augmentation):")
print(train_transform)

### 5.3 — Transforms de validacion y test (SIN augmentation)

Val y test NO reciben augmentation porque deben ser representativos del mundo real. Aplicar augmentation en evaluacion introduciria variabilidad artificial que haría las métricas poco confiables.

In [ ]:
# Transforms de val/test: solo Resize + CenterCrop determinista + Normalize
# IMPORTANTE: SIN augmentation
val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Pipeline de val/test (SIN augmentation):")
print(val_test_transform)

print("\nDiferencias clave:")
print("  Train  -> RandomResizedCrop + RandomHorizontalFlip + RandomRotation + ColorJitter")
print("  Val/Test -> CenterCrop (determinista), sin flips ni rotaciones")

## Sección 6 — DataLoaders

El `DataLoader` envuelve el `Dataset` y provee:
- **Batching:** agrupa imagenes en tensores de tamanio `BATCH_SIZE`
- **Shuffling:** en train, mezcla el orden de los datos en cada epoch para evitar que el modelo memorice la secuencia
- **Carga paralela:** `num_workers > 0` carga imagenes en background
- **`pin_memory`:** acelera la transferencia de datos a GPU si esta disponible
- **`drop_last=True`** solo en train: descarta el ultimo batch si es mas pequenio que `BATCH_SIZE`, evitando que BatchNorm tenga problemas con batches de 1 elemento

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 0  # Usar 0 en Windows para evitar errores de multiprocessing

train_dataset = AnimalDataset(DATA_DIR / "train.csv", transform=train_transform)
val_dataset   = AnimalDataset(DATA_DIR / "val.csv",   transform=val_test_transform)
test_dataset  = AnimalDataset(DATA_DIR / "test.csv",  transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,           # Solo en train
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,         # Solo en train
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # NO shuffle en val
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # NO shuffle en test
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Batches en train_loader: {len(train_loader)}")
print(f"Batches en val_loader:   {len(val_loader)}")
print(f"Batches en test_loader:  {len(test_loader)}")

## Sección 7 — Visualización del Efecto de las Augmentations

**Requerido explicitamente por la consigna.**

Se aplica `train_transform` 5 veces a la misma imagen base. Como las transformaciones son aleatorias (crop, flip, rotacion, color), cada aplicacion produce una version distinta. Esto simula la variabilidad que el modelo vera durante el entrenamiento.

Para visualizar correctamente, se desnormaliza el tensor antes de mostrarlo (se revierte la operacion `Normalize`).

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Revierte la normalizacion para visualizacion."""
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t  = torch.tensor(std).view(3, 1, 1)
    return (tensor * std_t + mean_t).clamp(0, 1)

In [ ]:
# Seleccionar una imagen representativa por clase (hasta 6 clases)
N_IMGS = min(len(raw_train.classes), 6)

selected = {}  # {clase: imagen PIL}
for i in range(len(raw_train)):
    img, lbl = raw_train[i]
    cls = raw_train.classes[lbl]
    if cls not in selected:
        selected[cls] = img
    if len(selected) == N_IMGS:
        break

N_AUGS = 5
n_rows = len(selected)
fig, axes = plt.subplots(n_rows, N_AUGS + 1, figsize=(3 * (N_AUGS + 1), 3 * n_rows))

if n_rows == 1:
    axes = axes[np.newaxis, :]

for row, (cls_name, orig_img) in enumerate(selected.items()):
    axes[row, 0].imshow(orig_img)
    axes[row, 0].set_title("Original", fontsize=8)
    axes[row, 0].set_ylabel(cls_name, fontsize=9, fontweight="bold")
    axes[row, 0].axis("off")

    for col in range(1, N_AUGS + 1):
        aug_tensor = train_transform(orig_img)
        aug_img    = denormalize(aug_tensor).permute(1, 2, 0).numpy()
        axes[row, col].imshow(aug_img)
        axes[row, col].set_title(f"Aug {col}", fontsize=8)
        axes[row, col].axis("off")

plt.suptitle("Efecto de Data Augmentation por clase (train_transform)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("augmentation_examples.png", dpi=100, bbox_inches="tight")
plt.show()
print("Guardado: dev/augmentation_examples.png")

**Observaciones de las augmentations:**

- `RandomResizedCrop`: se observan diferentes niveles de zoom y encuadre en cada version, simulando distintas distancias de la camara al animal.
- `RandomHorizontalFlip`: algunas versiones muestran el animal en espejo, lo cual es realista ya que los animales pueden orientarse en cualquier direccion.
- `RandomRotation(15)`: ligeras inclinaciones que replican imperfecciones de camara o posiciones del animal.
- `ColorJitter`: variaciones de brillo y saturacion que emulan diferentes condiciones de iluminacion (sol, sombra, horarios).

Estas transformaciones son fisicamente plausibles para imagenes de animales y no rompen la semantica de la imagen (no se usaron flips verticales ni rotaciones grandes).

## Sección 8 — Verificación Final del Batch

**Requerido explicitamente por la consigna.**

Se extrae un batch del `train_loader` para verificar que:
- El shape del tensor es correcto: `(BATCH_SIZE, 3, 224, 224)`
- El dtype de las imagenes es `float32` y el de las etiquetas es `int64`
- Los valores estan en el rango esperado despues de la normalizacion (media ~0, std ~1)
- Las etiquetas corresponden a las clases correctas

In [ ]:
# Metricas del batch
images, labels = next(iter(train_loader))

print("=" * 55)
print("VERIFICACION DEL BATCH DE ENTRENAMIENTO")
print("=" * 55)
print(f"\nShape del tensor de imagenes : {tuple(images.shape)}")
print(f"   (batch={images.shape[0]}, canales={images.shape[1]}, H={images.shape[2]}, W={images.shape[3]})")
print(f"\nShape del tensor de etiquetas: {tuple(labels.shape)}")
print(f"   dtype imagenes  : {images.dtype}")
print(f"   dtype etiquetas : {labels.dtype}")
print(f"\nRango de valores TRAS normalizacion:")
print(f"   min  = {images.min():.4f}")
print(f"   max  = {images.max():.4f}")
print(f"   mean = {images.mean():.4f}")
print(f"   std  = {images.std():.4f}")
print(f"\nEtiquetas en este batch: {labels.tolist()}")
print(f"Clases correspondientes: {[train_dataset.classes[l] for l in labels.tolist()]}")
print("=" * 55)

In [ ]:
# Visualizacion del batch desnormalizado
n_show = min(16, BATCH_SIZE)
ncols  = 8
nrows  = n_show // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, 2.8 * nrows))
axes = axes.flatten()

for i in range(n_show):
    img_vis = denormalize(images[i]).permute(1, 2, 0).numpy()
    axes[i].imshow(img_vis)
    axes[i].set_title(train_dataset.classes[labels[i].item()], fontsize=7)
    axes[i].axis("off")

plt.suptitle(
    f"Batch de train — {n_show} imagenes desnormalizadas (pares imagen-etiqueta verificados)",
    fontsize=11
)
plt.tight_layout()
plt.savefig("train_batch_sample.png", dpi=100, bbox_inches="tight")
plt.show()
print("Guardado: dev/train_batch_sample.png")

## Sección 9 — Resumen Final del Pipeline

Consolidacion de todos los parametros del pipeline de datos listos para la etapa de entrenamiento (Semana 3).

In [ ]:
print("=" * 60)
print("       RESUMEN COMPLETO DEL PIPELINE DE DATOS")
print("=" * 60)
print(f"\nDataset   : Team Separation (Animals) - Roboflow v5")
print(f"URL       : https://universe.roboflow.com/animals-67mq4/team-separation/dataset/5")
print(f"\nSplits:")
print(f"   Train : {len(train_dataset):>5} imagenes | {len(train_loader):>3} batches de {BATCH_SIZE}")
print(f"   Val   : {len(val_dataset):>5} imagenes")
print(f"   Test  : {len(test_dataset):>5} imagenes")
print(f"\nClases ({len(train_dataset.classes)}):")
for idx, cls in enumerate(train_dataset.classes):
    print(f"   {idx}: {cls}")
print(f"\nPreprocesamiento:")
print(f"   Modelo objetivo    : ResNet50 (ImageNet)")
print(f"   Tamanio de entrada : {IMG_SIZE}x{IMG_SIZE} px")
print(f"   Normalizacion mean : {IMAGENET_MEAN}")
print(f"   Normalizacion std  : {IMAGENET_STD}")
print(f"   Batch size         : {BATCH_SIZE}")
print(f"\nAugmentations (solo train):")
print(f"   OK RandomResizedCrop({IMG_SIZE}, scale=(0.8, 1.0))")
print(f"   OK RandomHorizontalFlip(p=0.5)")
print(f"   OK RandomRotation(15)")
print(f"   OK ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)")
print(f"\nVal/Test  : SIN augmentation — solo Resize + CenterCrop + Normalize")
print(f"Semilla   : {SEED}")
print("=" * 60)